# **Volve Field Trajectory Analysis — WITSML to Insight**

This notebook builds an end-to-end pipeline that pulls raw **WITSML** directional-survey
XML files for the [Volve field](https://www.equinor.com/energy/volve-data-sharing) (released
publicly by Equinor) out of a Databricks Volume, parses them into a tidy DataFrame, derives
real drilling-engineering metrics, and visualizes the well trajectories in 3D, plan view,
vertical section, and dogleg severity — the same views a directional driller would actually
use to QC a well path.

**Pipeline:** Databricks Volume (raw XML) → parse → engineer features (Dogleg Severity) → visualize → summarize.


## 1. Install and Import Dependencies

In [1]:
!pip install -q databricks-sdk welly bs4 plotly

import os
import glob
import numpy as np
import pandas as pd
from databricks.sdk import WorkspaceClient
from bs4 import BeautifulSoup
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Markdown, display

## 2. Configuration & Authentication



In [2]:
from google.colab import userdata

databricks_host = userdata.get('DATABRICKS_HOST')
databricks_token = userdata.get('DATABRICKS_TOKEN')

os.environ["DATABRICKS_HOST"] = databricks_host
os.environ["DATABRICKS_TOKEN"] = databricks_token

assert os.environ.get("DATABRICKS_HOST"), "Set DATABRICKS_HOST as an environment variable before running."
assert os.environ.get("DATABRICKS_TOKEN"), "Set DATABRICKS_TOKEN as an environment variable before running."

w = WorkspaceClient()

# Path Constants
BASE_VOLUME_PATH = "/Volumes/equinor_asa_volve_data_village/public/volve/WITSML Realtime drilling data"
WELLS = ['9-F-12', '9-F-14', '9-F-15', '9-F-11', '9-F-9', '9-F-7', '9-F-5', '9-F-4', '9-F-1 C']
PREFIX = "/Norway-Statoil-NO 15_"
SUFFIX = "/1/trajectory/1.xml"
LOCAL_DATA_DIR = "/content/wells_data"

# One consistent color per well, reused across every figure below
PALETTE = px.colors.qualitative.Bold
WELL_COLORS = {well: PALETTE[i % len(PALETTE)] for i, well in enumerate(WELLS)}

## 3. Helper Functions

In [3]:
def parse_witsml_to_df(xml_path, well_label):
    """Parses a WITSML XML file into a Pandas DataFrame including global metadata."""
    with open(xml_path, 'r') as f:
        soup = BeautifulSoup(f.read(), 'xml')

    traj = soup.find('trajectory')
    if not traj:
        return pd.DataFrame()

    # Capture global metadata (non-station tags)
    global_meta = {'well_id': well_label}
    for tag in traj.find_all(recursive=False):
        if tag.name != 'trajectoryStation':
            try:
                global_meta[tag.name] = float(tag.text)
            except ValueError:
                global_meta[tag.name] = tag.text

    # Extract individual trajectory stations
    rows = []
    for station in soup.find_all('trajectoryStation'):
        row = global_meta.copy()
        for tag in station.find_all():
            try:
                row[tag.name] = float(tag.text)
            except ValueError:
                row[tag.name] = tag.text
        rows.append(row)

    return pd.DataFrame(rows)

In [4]:
def compute_dogleg_severity(df, md_col='md', incl_col='incl', azi_col='azi'):
    """
    We add a 'dls' column (degrees / 30 m) to a single well's survey DataFrame.

    Using the standard minimum-curvature dogleg formula:
        cos(beta) = cos(incl2-incl1) - sin(incl1)*sin(incl2)*(1-cos(azi2-azi1))
        DLS = beta / (md2-md1) * 30
    """
    df = df.sort_values(md_col).reset_index(drop=True)
    if not {md_col, incl_col, azi_col}.issubset(df.columns):
        df['dls'] = np.nan
        return df

    incl = np.radians(df[incl_col].astype(float))
    azi = np.radians(df[azi_col].astype(float))
    md = df[md_col].astype(float)

    d_incl = incl.diff()
    d_azi = azi.diff()
    d_md = md.diff()

    cos_beta = (np.cos(d_incl) - np.sin(incl.shift()) * np.sin(incl) * (1 - np.cos(d_azi)))
    cos_beta = cos_beta.clip(-1, 1)
    beta_deg = np.degrees(np.arccos(cos_beta))

    with np.errstate(divide='ignore', invalid='ignore'):
        dls = np.where(d_md > 0, beta_deg / d_md * 30, np.nan)

    df['dls'] = dls
    return df

## 4. Data Acquisition (Download from Databricks)

In [5]:
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

for well in WELLS:
    remote_dir = f"{BASE_VOLUME_PATH}{PREFIX}$47$_{well}{SUFFIX}".rsplit('/', 1)[0]
    local_well_dir = os.path.join(LOCAL_DATA_DIR, well.replace(' ', '_'))
    os.makedirs(local_well_dir, exist_ok=True)

    try:
        items = list(w.files.list_directory_contents(remote_dir))
        for file_info in items:
            if not file_info.is_directory and file_info.path.endswith('.xml'):
                local_file = os.path.join(local_well_dir, os.path.basename(file_info.path))
                with open(local_file, 'wb') as f:
                    f.write(w.files.download(file_info.path).contents.read())
        print(f"\u2713 Processed well: {well}")
    except Exception as e:
        print(f"! Skip {well}: {e}")

✓ Processed well: 9-F-12
✓ Processed well: 9-F-14
✓ Processed well: 9-F-15
✓ Processed well: 9-F-11
✓ Processed well: 9-F-9
✓ Processed well: 9-F-7
✓ Processed well: 9-F-5
✓ Processed well: 9-F-4
✓ Processed well: 9-F-1 C


## 5. Data Processing (Batch Parsing + Feature Engineering)

In [6]:
all_dfs = []
xml_files = glob.glob(f'{LOCAL_DATA_DIR}/**/*.xml', recursive=True)

for file_path in xml_files:
    well_name = file_path.split('/')[-2]
    df_temp = parse_witsml_to_df(file_path, well_name)
    if not df_temp.empty:
        df_temp = compute_dogleg_severity(df_temp)
    all_dfs.append(df_temp)

master_df = pd.concat(all_dfs, ignore_index=True)
print(f"Final Master DataFrame: {master_df.shape[0]} rows, {master_df.shape[1]} columns across {master_df['well_id'].nunique()} wells.")
master_df.head()

Final Master DataFrame: 861 rows, 74 columns across 9 wells.


,well_id,nameWell,nameWellbore,name,dTimTrajStart,dTimTrajEnd,mdMn,mdMx,serviceCompany,magDeclUsed,...,sagIncCor,stnMagDeclUsed,valid,magTotalFieldCalc,magDipAngleCalc,gravTotalFieldCalc,objectGrowing,priv_userLastChange,priv_ipLastChange,priv_dTimReceived
0,9-F-11,NO 15/9-F-11,NO 15/9-F-11 T2,MWD-15/9-F-11 T2,2013-03-21T07:49:00.000Z,2013-05-04T05:52:42.000Z,0.0,4547.6,Baker Hughes INTEQ,-0.021817,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,9-F-11,NO 15/9-F-11,NO 15/9-F-11 T2,MWD-15/9-F-11 T2,2013-03-21T07:49:00.000Z,2013-05-04T05:52:42.000Z,0.0,4547.6,Baker Hughes INTEQ,-0.021817,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,9-F-11,NO 15/9-F-11,NO 15/9-F-11 T2,MWD-15/9-F-11 T2,2013-03-21T07:49:00.000Z,2013-05-04T05:52:42.000Z,0.0,4547.6,Baker Hughes INTEQ,-0.021817,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,9-F-11,NO 15/9-F-11,NO 15/9-F-11 T2,MWD-15/9-F-11 T2,2013-03-21T07:49:00.000Z,2013-05-04T05:52:42.000Z,0.0,4547.6,Baker Hughes INTEQ,-0.021817,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,9-F-11,NO 15/9-F-11,NO 15/9-F-11 T2,MWD-15/9-F-11 T2,2013-03-21T07:49:00.000Z,2013-05-04T05:52:42.000Z,0.0,4547.6,Baker Hughes INTEQ,-0.021817,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 6. Visualization Dashboard

Four complementary views of the same survey data, mirroring how directional drillers
actually QC a well path: a 3D hero shot, a plan-view fan-out from the platform, a
vertical section (the industry-standard 2D profile), and dogleg severity to flag
aggressive curvature.


In [7]:
fig = go.Figure()

for well_id, grp in master_df.groupby('well_id'):
    grp = grp.sort_values('md') if 'md' in grp.columns else grp
    has_survey_cols = {'md', 'incl', 'azi'}.issubset(grp.columns)
    fig.add_trace(go.Scatter3d(
        x=grp['dispNs'], y=grp['dispEw'], z=grp['tvd'],
        mode='lines',
        name=well_id,
        line=dict(width=5, color=WELL_COLORS.get(well_id)),
        hovertemplate=(
            f"<b>{well_id}</b><br>"
            "MD: %{customdata[0]:.0f} m<br>"
            "Incl: %{customdata[1]:.1f}\u00b0<br>"
            "Azi: %{customdata[2]:.1f}\u00b0<extra></extra>"
        ) if has_survey_cols else None,
        customdata=grp[['md', 'incl', 'azi']].values if has_survey_cols else None,
    ))
    # Mark the surface location (start) for each well
    fig.add_trace(go.Scatter3d(
        x=[grp['dispNs'].iloc[0]], y=[grp['dispEw'].iloc[0]], z=[grp['tvd'].iloc[0]],
        mode='markers', marker=dict(size=4, color=WELL_COLORS.get(well_id)),
        showlegend=False, hoverinfo='skip'
    ))

fig.update_scenes(
    zaxis_autorange="reversed",  # depth increases downward
    xaxis_title="North-South Displacement (m)",
    yaxis_title="East-West Displacement (m)",
    zaxis_title="TVD (m)",
)
fig.update_layout(
    title="Volve Field \u2014 3D Well Trajectories",
    template="plotly_dark",
    height=700,
    legend_title="Well",
)
fig.show()

In [8]:
fig = px.line(
    master_df.sort_values('md') if 'md' in master_df.columns else master_df,
    x='dispEw', y='dispNs', color='well_id',
    color_discrete_map=WELL_COLORS,
    title="Plan View \u2014 Well Paths Fanning Out From Platform",
    labels={'dispEw': 'East-West Displacement (m)', 'dispNs': 'North-South Displacement (m)'},
)
fig.update_yaxes(scaleanchor="x", scaleratio=1)  # true-to-scale map
fig.update_layout(template="plotly_dark", height=600)
fig.show()

In [9]:
master_df['horiz_disp'] = np.sqrt(master_df['dispNs']**2 + master_df['dispEw']**2)

fig = px.line(
    master_df.sort_values('md') if 'md' in master_df.columns else master_df,
    x='horiz_disp', y='tvd', color='well_id',
    color_discrete_map=WELL_COLORS,
    title="Vertical Section \u2014 Depth vs. Horizontal Reach",
    labels={'horiz_disp': 'Horizontal Displacement (m)', 'tvd': 'TVD (m)'},
)
fig.update_yaxes(autorange="reversed")
fig.update_layout(template="plotly_dark", height=600)
fig.show()

In [10]:
if 'dls' in master_df.columns and master_df['dls'].notna().any():
    wells_sorted = sorted(master_df['well_id'].unique())
    fig = make_subplots(rows=len(wells_sorted), cols=1, shared_xaxes=True,
                         subplot_titles=wells_sorted, vertical_spacing=0.02)

    for i, well_id in enumerate(wells_sorted, start=1):
        grp = master_df[master_df['well_id'] == well_id].sort_values('md')
        fig.add_trace(
            go.Scatter(x=grp['md'], y=grp['dls'], mode='lines',
                       line=dict(color=WELL_COLORS.get(well_id)), showlegend=False),
            row=i, col=1
        )
        # 3 deg/30m is a common practical DLS caution threshold
        fig.add_hline(y=3, line_dash="dot", line_color="red", opacity=0.4, row=i, col=1)

    fig.update_layout(
        title="Dogleg Severity by Measured Depth (red line = 3\u00b0/30m caution threshold)",
        template="plotly_dark", height=180 * len(wells_sorted),
    )
    fig.update_xaxes(title_text="Measured Depth (m)", row=len(wells_sorted), col=1)
    fig.show()
else:
    print("Skipping DLS plot: 'incl'/'azi'/'md' columns not found in parsed survey data.")

## 7. Summary

In [11]:
summary = (
    master_df.groupby('well_id')
    .agg(max_md=('md', 'max') if 'md' in master_df.columns else ('tvd', 'max'),
         max_tvd=('tvd', 'max'),
         max_horiz_reach=('horiz_disp', 'max'),
         max_dls=('dls', 'max') if 'dls' in master_df.columns else ('tvd', 'max'),
         avg_dls=('dls', 'mean') if 'dls' in master_df.columns else ('tvd', 'max'))
    .round(1)
    .sort_values('max_tvd', ascending=False)
)

deepest = summary['max_tvd'].idxmax()
longest_reach = summary['max_horiz_reach'].idxmax()
most_severe = summary['max_dls'].idxmax() if summary['max_dls'].notna().any() else None

dls_line = ""
if most_severe:
    dls_line = f"- **Most aggressive curvature:** `{most_severe}`, peak DLS of **{summary.loc[most_severe, 'max_dls']:.1f} deg/30m**."

report = f"""
### Key figures (computed from this run)

- **{len(summary)} wells** processed, spanning a max TVD range of
  **{summary['max_tvd'].min():.0f}-{summary['max_tvd'].max():.0f} m**.
- **Deepest well:** `{deepest}` at **{summary.loc[deepest, 'max_tvd']:.0f} m TVD**.
- **Longest horizontal reach:** `{longest_reach}` at **{summary.loc[longest_reach, 'max_horiz_reach']:.0f} m**
  from the platform.
{dls_line}

Full per-well table:
"""

display(Markdown(report))
summary


### Key figures (computed from this run)

- **9 wells** processed, spanning a max TVD range of
  **1013-3384 m**.
- **Deepest well:** `9-F-11` at **3384 m TVD**.
- **Longest horizontal reach:** `9-F-5` at **1361 m**
  from the platform.
- **Most aggressive curvature:** `9-F-11`, peak DLS of **0.1 deg/30m**.

Full per-well table:


,max_md,max_tvd,max_horiz_reach,max_dls,avg_dls
well_id,,,,,
9-F-11,4547.6,3384.3,1251.1,0.1,0.0
9-F-1_C,3453.8,3248.7,830.4,0.1,0.0
9-F-5,3792.0,3246.4,1360.6,0.1,0.0
9-F-14,3750.0,3158.7,1025.9,0.1,0.0
9-F-4,3510.0,3138.1,1070.4,0.0,0.0
9-F-12,3520.0,3108.4,463.0,0.1,0.0
9-F-15,3232.0,3044.1,754.9,0.1,0.0
9-F-7,1083.0,1077.5,79.4,0.0,0.0
9-F-9,1206.0,1013.0,450.9,0.1,0.0
